# Lab 1 — сравнение LLM для проекта «Планировщик питания»

**Модели:** GigaChat 2 Lite, Qwen3-4B, Mistral Small (`mistral-small-latest`).

Цель notebook — провести одинаковый эксперимент на трёх сценариях, проверить структурированный JSON, ограничения, использование продуктов из домашнего запаса и latency. Результаты не задаются заранее.

**Главное изменение этой версии:** результаты сохраняются после каждого отдельного прогона в checkpoint CSV, чтобы завершение/перезапуск Colab не уничтожил уже полученные данные.


## 0. Среда

Qwen3-4B запускается локально через Transformers и требует GPU. GigaChat и Mistral работают через API.

Для Qwen время загрузки модели измеряется отдельно и не входит в пользовательскую latency генерации.

Рекомендуется после запуска notebook сохранять checkpoint CSV на Google Drive или скачивать его после каждого завершённого блока модели.


In [ ]:
!pip -q install gigachat mistralai transformers accelerate sentencepiece pydantic pandas


In [ ]:
import os
import json
import re
import time
import gc
from pathlib import Path
from typing import List

import pandas as pd
import torch
from pydantic import BaseModel, Field
from google.colab import userdata

try:
    GIGACHAT_CREDENTIALS = userdata.get("GIGACHAT_CREDENTIALS")
except Exception:
    GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")

try:
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except Exception:
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("GigaChat credentials configured:", bool(GIGACHAT_CREDENTIALS))
print("Mistral API key configured:", bool(MISTRAL_API_KEY))


CUDA available: True
GPU: Tesla T4
GigaChat credentials configured: True
Mistral API key configured: True


## 1. Checkpoint результатов

Этот блок создаёт файл `lab1_results_checkpoint.csv` и функции для сохранения **после каждого завершённого вызова**.

Если доступен Google Drive, можно указать путь в `CHECKPOINT_PATH` на Drive. Иначе используется `/content/`, и файл нужно периодически скачивать вручную.


In [ ]:
# По умолчанию checkpoint находится в текущей сессии Colab.
# Для постоянного хранения можно заранее подключить Google Drive и заменить путь, например:
# CHECKPOINT_PATH = "/content/drive/MyDrive/AI_lab1/lab1_results_checkpoint.csv"
CHECKPOINT_PATH = "/content/lab1_results_checkpoint.csv"

RESULT_COLUMNS = [
    "model", "scenario", "repeat", "status", "latency_sec",
    "model_load_sec", "input_tokens", "output_tokens", "total_tokens",
    "json_valid", "schema_valid", "forbidden_hits", "pantry_mentions",
    "within_30_sec", "error", "raw_output"
]

if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)
    print("Найден существующий checkpoint:", len(results_df), "строк")
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    print("Создан новый checkpoint")

def save_checkpoint():
    global results_df
    results_df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

def append_result(row):
    global results_df
    safe_row = {col: row.get(col, None) for col in RESULT_COLUMNS}
    results_df = pd.concat([results_df, pd.DataFrame([safe_row])], ignore_index=True)
    save_checkpoint()
    print(
        f"Сохранено: {row.get('model')} | {row.get('scenario')} | "
        f"repeat={row.get('repeat')} | status={row.get('status')} | "
        f"всего строк={len(results_df)}"
    )

print("Checkpoint:", CHECKPOINT_PATH)


Создан новый checkpoint
Checkpoint: /content/lab1_results_checkpoint.csv


## 2. Общий prompt и тестовые сценарии

Один и тот же prompt используется всеми тремя моделями.


In [ ]:
SYSTEM_PROMPT = """Ты — AI-помощник приложения «Планировщик питания».
Составь практичный план питания на указанное число дней.

Правила:
1. Соблюдай цель, бюджет, ограничения и максимальное время приготовления.
2. Не используй запрещённые продукты.
3. Старайся использовать продукты из pantry (домашнего запаса).
4. Старайся повторно использовать ингредиенты и уменьшать остатки.
5. Не выдумывай точную стоимость и точные КБЖУ: они рассчитываются отдельными программными инструментами.
6. Ответ должен быть только валидным JSON без Markdown.

Формат JSON:
{
  "days": [
    {
      "day": 1,
      "meals": [
        {
          "name": "название блюда",
          "meal_type": "breakfast|lunch|dinner|snack",
          "ingredients": [
            {"name": "продукт", "amount_g": 100}
          ],
          "cook_time_min": 20
        }
      ]
    }
  ],
  "notes": "краткие пояснения"
}"""

SCENARIOS = [
    {
        "id": "A_basic",
        "user": "Студент, цель — снижение веса. Бюджет 2500 ₽ в неделю. Нельзя рыбу. Дома есть гречка, яйца и курица. Составь меню на 3 дня. Максимальное время приготовления — 30 минут."
    },
    {
        "id": "B_budget",
        "user": "Молодой специалист, цель — поддержание веса. Бюджет 2500 ₽ в неделю. Дома есть морковь, курица и гречка. Составь меню на 3 дня, стараясь не покупать лишнего и использовать домашние продукты."
    },
    {
        "id": "C_zero_waste",
        "user": "Составь меню на 3 дня с минимизацией отходов. Дома есть 500 г моркови, 400 г курицы и 1000 г гречки. Нужно максимально использовать эти продукты и не создавать большие остатки."
    }
]

print("Сценариев:", len(SCENARIOS))


Сценариев: 3


## 3. Схема и валидация результата


In [ ]:
class Ingredient(BaseModel):
    name: str
    amount_g: float = Field(ge=0)

class Meal(BaseModel):
    name: str
    meal_type: str
    ingredients: List[Ingredient]
    cook_time_min: float = Field(ge=0)

class DayPlan(BaseModel):
    day: int
    meals: List[Meal]

class MealPlan(BaseModel):
    days: List[DayPlan]
    notes: str = ""

def extract_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        if not match:
            raise
        return json.loads(match.group(0))

def validate_plan(text, scenario):
    result = {
        "json_valid": False,
        "schema_valid": False,
        "forbidden_hits": 0,
        "pantry_mentions": 0,
        "error": ""
    }
    try:
        data = extract_json(text)
        result["json_valid"] = True
        plan = MealPlan.model_validate(data)
        result["schema_valid"] = True
        names = " ".join(
            item.name.lower()
            for day in plan.days
            for meal in day.meals
            for item in meal.ingredients
        )
        forbidden = ["рыб"] if scenario["id"] == "A_basic" else []
        result["forbidden_hits"] = sum(word in names for word in forbidden)
        pantry_terms = ["греч", "яйц", "куриц", "морков"]
        result["pantry_mentions"] = sum(word in names for word in pantry_terms)
    except Exception as exc:
        result["error"] = str(exc)[:300]
    return result


## 4. Вспомогательные функции


In [ ]:
def usage_value(usage, key):
    if usage is None:
        return None
    if isinstance(usage, dict):
        return usage.get(key)
    return getattr(usage, key, None)

def normalize_usage(usage):
    return {
        "input_tokens": usage_value(usage, "prompt_tokens") or usage_value(usage, "input_tokens"),
        "output_tokens": usage_value(usage, "completion_tokens") or usage_value(usage, "output_tokens"),
        "total_tokens": usage_value(usage, "total_tokens")
    }

def completed_repeats(model_name, scenario_id):
    if len(results_df) == 0:
        return set()
    mask = (results_df["model"] == model_name) & (results_df["scenario"] == scenario_id)
    return set(pd.to_numeric(results_df.loc[mask, "repeat"], errors="coerce").dropna().astype(int).tolist())

def run_model_checkpointed(model_name, call_fn, repeats=3, model_load_sec=None):
    total = len(SCENARIOS) * repeats
    done = 0
    for scenario in SCENARIOS:
        already = completed_repeats(model_name, scenario["id"])
        for repeat in range(1, repeats + 1):
            if repeat in already:
                print(f"Пропуск уже сохранённого: {model_name} / {scenario['id']} / repeat {repeat}")
                done += 1
                continue
            print(f"\n[{done+1}/{total}] {model_name} | {scenario['id']} | repeat {repeat}")
            start = time.perf_counter()
            try:
                text, latency, usage = call_fn(scenario["user"])
                checks = validate_plan(text, scenario)
                u = normalize_usage(usage)
                row = {
                    "model": model_name,
                    "scenario": scenario["id"],
                    "repeat": repeat,
                    "status": "ok",
                    "latency_sec": round(float(latency), 3),
                    "model_load_sec": model_load_sec,
                    **u,
                    **checks,
                    "within_30_sec": float(latency) <= 30,
                    "error": "",
                    "raw_output": text
                }
            except Exception as exc:
                row = {
                    "model": model_name,
                    "scenario": scenario["id"],
                    "repeat": repeat,
                    "status": "error",
                    "latency_sec": None,
                    "model_load_sec": model_load_sec,
                    "error": repr(exc)[:500],
                    "raw_output": ""
                }
            append_result(row)
            done += 1
    print(f"Завершено для {model_name}: {done}/{total}")


## 5. GigaChat 2 Lite

В актуальном API Sber для GigaChat 2 Lite используются идентификаторы `GigaChat` / `GigaChat-2`; `GigaChat-2-Lite` использовать не нужно.


In [ ]:
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

giga = None
if GIGACHAT_CREDENTIALS:
    giga = GigaChat(
        credentials=GIGACHAT_CREDENTIALS,
        model="GigaChat-2",
        verify_ssl_certs=False
    )
    print("GigaChat client: OK")
else:
    print("GigaChat пропущен: не задан GIGACHAT_CREDENTIALS")

def call_gigachat(user_text):
    if giga is None:
        raise RuntimeError("GIGACHAT_CREDENTIALS не настроен")
    messages = [
        Messages(role=MessagesRole.SYSTEM, content=SYSTEM_PROMPT),
        Messages(role=MessagesRole.USER, content=user_text)
    ]
    start = time.perf_counter()
    response = giga.chat(Chat(messages=messages))
    latency = time.perf_counter() - start
    text = response.choices[0].message.content
    usage = getattr(response, "usage", None)
    return text, latency, usage

if giga is not None:
    text, latency, usage = call_gigachat(SCENARIOS[0]["user"])
    print("Smoke-test latency:", round(latency, 2), "sec")
    print(text[:1500])


GigaChat client: OK
Smoke-test latency: 6.28 sec
{
  "days": [
    {
      "day": 1,
      "meals": [
        {
          "name": "Омлет с помидорами",
          "meal_type": "breakfast",
          "ingredients": [
            {"name": "Яйца", "amount_g": 200},
            {"name": "Помидоры", "amount_g": 200}
          ],
          "cook_time_min": 10
        },
        {
          "name": "Гречневая каша",
          "meal_type": "lunch",
          "ingredients": [
            {"name": "Гречка", "amount_g": 100}
          ],
          "cook_time_min": 15
        },
        {
          "name": "Куриная грудка с овощами",
          "meal_type": "dinner",
          "ingredients": [
            {"name": "Куриная грудка", "amount_g": 150},
            {"name": "Кабачок", "amount_g": 100},
            {"name": "Болгарский перец", "amount_g": 100}
          ],
          "cook_time_min": 20
        }
      ]
    },
    {
      "day": 2,
      "meals": [
        {
          "name": "Творог с я

## 6. Mistral Small API

Используется официальный Chat Completions API. В Colab Secret создайте `MISTRAL_API_KEY`.

В эксперименте указан `mistral-small-latest`, который используется в официальном quickstart Mistral API.


In [ ]:
!pip install -U "mistralai>=2"

In [ ]:
import os
import requests

url = "https://api.mistral.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {MISTRAL_API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "mistral-small-latest",
    "messages": [
        {"role": "user", "content": "Ответь одним словом: привет"}
    ],
    "max_tokens": 20
}

response = requests.post(url, headers=headers, json=payload)

print("HTTP status:", response.status_code)
print("Headers:")
for key in ["x-ratelimit-limit", "x-ratelimit-remaining", "x-ratelimit-reset"]:
    print(key, "=", response.headers.get(key))

print("Body:")
print(response.text[:2000])

HTTP status: 429
Headers:
x-ratelimit-limit = None
x-ratelimit-remaining = None
x-ratelimit-reset = None
Body:
{"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}


In [ ]:
from mistralai.client import Mistral

mistral = None
if MISTRAL_API_KEY:
    mistral = Mistral(api_key=MISTRAL_API_KEY)
    print("Mistral client: OK")
else:
    print("Mistral пропущен: не задан MISTRAL_API_KEY")

MISTRAL_MODEL = "mistral-small-latest"

def call_mistral(user_text):
    if mistral is None:
        raise RuntimeError("MISTRAL_API_KEY не настроен")
    start = time.perf_counter()
    response = mistral.chat.complete(
        model=MISTRAL_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text}
        ],
        temperature=0
    )
    latency = time.perf_counter() - start
    text = response.choices[0].message.content
    usage = getattr(response, "usage", None)
    return text, latency, usage

if mistral is not None:
    text, latency, usage = call_mistral(SCENARIOS[0]["user"])
    print("Smoke-test latency:", round(latency, 2), "sec")
    print(text[:1500])


Mistral client: OK


SDKError: API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

## 7. Qwen3-4B

Qwen — единственная локальная модель в новом эксперименте. Важное отличие от предыдущей версии: после каждой генерации строка сохраняется в checkpoint.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_ID = "Qwen/Qwen3-4B"
qwen_tokenizer = None
qwen_model = None
qwen_load_sec = None

if not torch.cuda.is_available():
    print("ВНИМАНИЕ: GPU не обнаружен. Полный Qwen-прогон на CPU запускать не рекомендуется.")
else:
    try:
        load_start = time.perf_counter()
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ID)
        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_ID,
            torch_dtype="auto",
            device_map="auto"
        )
        qwen_load_sec = time.perf_counter() - load_start
        print("Qwen loaded in", round(qwen_load_sec, 2), "sec")
    except Exception as exc:
        print("Qwen load error:", repr(exc))

def call_qwen(user_text, max_new_tokens=1400):
    if qwen_model is None or qwen_tokenizer is None:
        raise RuntimeError("Qwen не загружен")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]
    prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)
    start = time.perf_counter()
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )
    latency = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = qwen_tokenizer.decode(generated, skip_special_tokens=True)
    input_tokens = int(inputs["input_ids"].shape[1])
    output_tokens = int(generated.shape[0])
    usage = {
        "prompt_tokens": input_tokens,
        "completion_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens
    }
    return text, latency, usage

if qwen_model is not None:
    text, latency, usage = call_qwen(SCENARIOS[0]["user"])
    print("Smoke-test latency:", round(latency, 2), "sec")
    print("Tokens:", usage)
    print(text[:1500])


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen loaded in 370.45 sec
Smoke-test latency: 65.3 sec
Tokens: {'prompt_tokens': 366, 'completion_tokens': 947, 'total_tokens': 1313}
{
  "days": [
    {
      "day": 1,
      "meals": [
        {
          "name": "Овсяная каша с яйцом",
          "meal_type": "breakfast",
          "ingredients": [
            {"name": "овсяные хлопья", "amount_g": 40},
            {"name": "молоко", "amount_g": 100},
            {"name": "яйцо", "amount_g": 50}
          ],
          "cook_time_min": 15
        },
        {
          "name": "Гречка с курицей",
          "meal_type": "lunch",
          "ingredients": [
            {"name": "гречка", "amount_g": 100},
            {"name": "курица", "amount_g": 150}
          ],
          "cook_time_min": 25
        },
        {
          "name": "Салат с яйцом",
          "meal_type": "dinner",
          "ingredients": [
            {"name": "яйцо", "amount_g": 50},
            {"name": "огурец", "amount_g": 100},
            {"name": "лук", "amount_

## 8. Полный эксперимент

Порядок запуска: **GigaChat → Mistral → Qwen**.

Каждая завершённая генерация сразу записывается в checkpoint. Если Colab прервётся, после повторного запуска notebook существующие строки будут пропущены.


In [ ]:
# 8.1 GigaChat — 9 вызовов
if giga is not None:
    run_model_checkpointed("GigaChat 2 Lite", call_gigachat, repeats=3)
else:
    print("GigaChat пропущен")



[1/9] GigaChat 2 Lite | A_basic | repeat 1


/tmp/ipykernel_11730/17554276.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([safe_row])], ignore_index=True)


Сохранено: GigaChat 2 Lite | A_basic | repeat=1 | status=ok | всего строк=1

[2/9] GigaChat 2 Lite | A_basic | repeat 2
Сохранено: GigaChat 2 Lite | A_basic | repeat=2 | status=ok | всего строк=2

[3/9] GigaChat 2 Lite | A_basic | repeat 3
Сохранено: GigaChat 2 Lite | A_basic | repeat=3 | status=ok | всего строк=3

[4/9] GigaChat 2 Lite | B_budget | repeat 1
Сохранено: GigaChat 2 Lite | B_budget | repeat=1 | status=ok | всего строк=4

[5/9] GigaChat 2 Lite | B_budget | repeat 2
Сохранено: GigaChat 2 Lite | B_budget | repeat=2 | status=ok | всего строк=5

[6/9] GigaChat 2 Lite | B_budget | repeat 3
Сохранено: GigaChat 2 Lite | B_budget | repeat=3 | status=ok | всего строк=6

[7/9] GigaChat 2 Lite | C_zero_waste | repeat 1
Сохранено: GigaChat 2 Lite | C_zero_waste | repeat=1 | status=ok | всего строк=7

[8/9] GigaChat 2 Lite | C_zero_waste | repeat 2
Сохранено: GigaChat 2 Lite | C_zero_waste | repeat=2 | status=ok | всего строк=8

[9/9] GigaChat 2 Lite | C_zero_waste | repeat 3
Сохранено

In [ ]:
# 8.2 Mistral — 9 вызовов
if mistral is not None:
    run_model_checkpointed("Mistral Small", call_mistral, repeats=3)
else:
    print("Mistral пропущен")


In [ ]:
# 8.3 Qwen — 9 вызовов
if qwen_model is not None:
    run_model_checkpointed("Qwen3-4B", call_qwen, repeats=3, model_load_sec=qwen_load_sec)
else:
    print("Qwen пропущен")



[1/9] Qwen3-4B | A_basic | repeat 1


/tmp/ipykernel_11730/17554276.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([safe_row])], ignore_index=True)


Сохранено: Qwen3-4B | A_basic | repeat=1 | status=ok | всего строк=10

[2/9] Qwen3-4B | A_basic | repeat 2
Сохранено: Qwen3-4B | A_basic | repeat=2 | status=ok | всего строк=11

[3/9] Qwen3-4B | A_basic | repeat 3
Сохранено: Qwen3-4B | A_basic | repeat=3 | status=ok | всего строк=12

[4/9] Qwen3-4B | B_budget | repeat 1
Сохранено: Qwen3-4B | B_budget | repeat=1 | status=ok | всего строк=13

[5/9] Qwen3-4B | B_budget | repeat 2
Сохранено: Qwen3-4B | B_budget | repeat=2 | status=ok | всего строк=14

[6/9] Qwen3-4B | B_budget | repeat 3
Сохранено: Qwen3-4B | B_budget | repeat=3 | status=ok | всего строк=15

[7/9] Qwen3-4B | C_zero_waste | repeat 1
Сохранено: Qwen3-4B | C_zero_waste | repeat=1 | status=ok | всего строк=16

[8/9] Qwen3-4B | C_zero_waste | repeat 2
Сохранено: Qwen3-4B | C_zero_waste | repeat=2 | status=ok | всего строк=17

[9/9] Qwen3-4B | C_zero_waste | repeat 3
Сохранено: Qwen3-4B | C_zero_waste | repeat=3 | status=ok | всего строк=18
Завершено для Qwen3-4B: 9/9


In [ ]:
# Быстрый просмотр текущего checkpoint
results_df = pd.read_csv(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else pd.DataFrame(columns=RESULT_COLUMNS)
print("Всего сохранённых записей:", len(results_df))
if len(results_df):
    display(results_df[[
        "model", "scenario", "repeat", "status", "latency_sec",
        "input_tokens", "output_tokens", "total_tokens",
        "json_valid", "schema_valid", "forbidden_hits", "pantry_mentions",
        "within_30_sec"
    ]])


Всего сохранённых записей: 18


,model,scenario,repeat,status,latency_sec,input_tokens,output_tokens,total_tokens,json_valid,schema_valid,forbidden_hits,pantry_mentions,within_30_sec
0,GigaChat 2 Lite,A_basic,1,ok,6.146,296,724,1020,True,True,0,2,True
1,GigaChat 2 Lite,A_basic,2,ok,4.460,296,744,1040,True,True,0,3,True
2,GigaChat 2 Lite,A_basic,3,ok,4.470,59,746,805,True,True,0,3,True
3,GigaChat 2 Lite,B_budget,1,ok,3.696,184,514,698,True,True,0,2,True
4,GigaChat 2 Lite,B_budget,2,ok,3.167,293,506,799,True,True,0,2,True
5,GigaChat 2 Lite,B_budget,3,ok,3.108,293,505,798,True,True,0,2,True
6,GigaChat 2 Lite,C_zero_waste,1,ok,3.279,186,510,696,True,False,0,0,True
7,GigaChat 2 Lite,C_zero_waste,2,ok,3.116,186,495,681,True,True,0,3,True
8,GigaChat 2 Lite,C_zero_waste,3,ok,3.446,58,531,589,True,True,0,3,True
9,Qwen3-4B,A_basic,1,ok,65.623,366,947,1313,True,True,0,4,False


## 9. Сводные метрики


In [ ]:
results_df = pd.read_csv(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else pd.DataFrame(columns=RESULT_COLUMNS)
if len(results_df):
    tmp = results_df.copy()
    tmp["latency_sec"] = pd.to_numeric(tmp["latency_sec"], errors="coerce")
    tmp["total_tokens"] = pd.to_numeric(tmp["total_tokens"], errors="coerce")
    tmp["json_valid_num"] = tmp["json_valid"].fillna(False).astype(str).str.lower().eq("true")
    tmp["schema_valid_num"] = tmp["schema_valid"].fillna(False).astype(str).str.lower().eq("true")
    tmp["within_30_num"] = tmp["within_30_sec"].fillna(False).astype(str).str.lower().eq("true")
    summary = tmp.groupby("model").agg(
        runs=("status", "count"),
        successful_runs=("status", lambda s: (s == "ok").sum()),
        avg_latency_sec=("latency_sec", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        json_valid_rate=("json_valid_num", "mean"),
        schema_valid_rate=("schema_valid_num", "mean"),
        avg_forbidden_hits=("forbidden_hits", "mean"),
        avg_pantry_mentions=("pantry_mentions", "mean"),
        within_30_sec_rate=("within_30_num", "mean")
    ).reset_index()
    display(summary)
else:
    print("Нет результатов.")


,model,runs,successful_runs,avg_latency_sec,avg_total_tokens,json_valid_rate,schema_valid_rate,avg_forbidden_hits,avg_pantry_mentions,within_30_sec_rate
0,GigaChat 2 Lite,9,9,3.876444,791.777778,1.0,0.888889,0.0,2.222222,1.0
1,Qwen3-4B,9,9,64.370222,1297.666667,1.0,1.000000,0.0,3.666667,0.0


## 10. Инструмент: расчёт стоимости

Это демонстрация принципа **LLM + tool**. В production `PRICE_DB` должна быть заменена на БД продуктов и актуальных цен.


In [ ]:
PRICE_DB = {
    "гречка": 120.0,
    "яйца": 110.0,
    "курица": 320.0,
    "морковь": 60.0,
    "рис": 100.0,
    "овсянка": 90.0,
}

def calculate_cost(items):
    total = 0.0
    details = []
    for item in items:
        name = item["name"].lower()
        amount_g = float(item["amount_g"])
        matched = next((key for key in PRICE_DB if key in name), None)
        if matched is None:
            details.append({"name": item["name"], "amount_g": amount_g, "cost": None})
            continue
        cost = PRICE_DB[matched] * amount_g / 1000
        total += cost
        details.append({"name": item["name"], "amount_g": amount_g, "cost": round(cost, 2)})
    return round(total, 2), details

example_items = [
    {"name": "гречка", "amount_g": 500},
    {"name": "курица", "amount_g": 400},
    {"name": "морковь", "amount_g": 300},
]

print(calculate_cost(example_items))


(206.0, [{'name': 'гречка', 'amount_g': 500.0, 'cost': 60.0}, {'name': 'курица', 'amount_g': 400.0, 'cost': 128.0}, {'name': 'морковь', 'amount_g': 300.0, 'cost': 18.0}])


## 11. Источники

- GigaChat models: https://developers.sber.ru/docs/ru/gigachat/models/gigachat-2-lite
- GigaChat functions: https://developers.sber.ru/docs/ru/gigachat/guides/functions/overview
- GigaChat tariffs: https://developers.sber.ru/docs/ru/gigachat/tariffs/individual-tariffs
- Mistral models: https://docs.mistral.ai/models
- Mistral API quickstart: https://docs.mistral.ai/getting-started/quickstarts/studio/activate-and-generate-api-key
- Mistral pricing: https://docs.mistral.ai/inference/pricing
- Mistral function calling: https://docs.mistral.ai/studio/conversations/function-calling
- Qwen3-4B: https://huggingface.co/Qwen/Qwen3-4B
